# Inspect HKS normalization

This notebook smooths one organoid mesh with a low-pass Laplace-Beltrami reconstruction, then plots HKS curves and mean-based normalizations for sampled mesh vertices.

## HKS setup

`scripts/run_crypt_segmentation.py` loads each mesh, normalizes coordinates, computes the Laplace-Beltrami eigendecomposition, then calls `segment_crypts_organoid`. This notebook keeps the same mesh and HKS utilities, but uses a wider hand-chosen HKS time grid for normalization experiments.

The mesh is first smoothed by projecting vertex coordinates into the Laplace-Beltrami eigenbasis and reconstructing them from the first `l_max ** 2` modes with `l_max = 12`. HKS is then recomputed on this smoothed geometry for log-spaced times from `0.1` to `1000`.

The curve panels show only mean-based quantities, avoiding median and quantile transforms at large times where all vertices approach a common surface-area-dependent HKS value.

In [120]:
import heapq
import igl
from pathlib import Path

import numpy as np
from IPython.display import display
import plotly.graph_objects as go
from plotly import colors as plotly_colors
from plotly.subplots import make_subplots
from scipy.optimize import minimize

from organograph.io_utils.dataset_config import load_mesh_dataset_config
from organograph.mesh.OrganoidMesh import OrganoidMesh
from organograph.mesh.hks import compute_hks
from organograph.plotting.meshes import plot_mesh_by_regions, plot_organoid_mesh


In [ ]:
# --- Organoid selection ---
NOTEBOOK_DIR = Path.cwd().resolve()
if NOTEBOOK_DIR.name != "notebooks" and (NOTEBOOK_DIR / "notebooks").exists():
    NOTEBOOK_DIR = NOTEBOOK_DIR / "notebooks"
PROJECT_ROOT = NOTEBOOK_DIR.parent

DATASET = "20251201"
TIMEPOINT = "day4p5"
WELL = "B02"
ORGANOID_ID = "124"

#     {"timepoint": "day4p5", "well": "B03", "organoid_id": "144"},
#     {"timepoint": "day4p5", "well": "B02", "organoid_id": "124"},
#     {"timepoint": "day4p5", "well": "B04", "organoid_id": "4"},
#     {"timepoint": "day4p5", "well": "B05", "organoid_id": "54"},
#     {"timepoint": "day4p5", "well": "B02", "organoid_id": "115"},
#     {"timepoint": "day4p5", "well": "B02", "organoid_id": "100"},
#     {"timepoint": "day4p5", "well": "B02", "organoid_id": "31"},
#     {"timepoint": "day4p5", "well": "B02", "organoid_id": "39"},

MESH_DATA_DIR = PROJECT_ROOT.parent / "NicoleData" / DATASET / "fractal_output"
MESH_CONFIG_PATH = PROJECT_ROOT.parent / "NicoleData" / DATASET / "mesh_config.json"
N_SAMPLE_POINTS = 25
SAMPLE_RANDOM_SEED = 7
EIGENMODES = 225
SMOOTH_LMAX = 12
HKS_T_MIN = 0.5
HKS_T_MAX = 1000.0
HKS_N_TIMES = 60

mesh_cfg = load_mesh_dataset_config(str(MESH_CONFIG_PATH))

mesh_path = (
    MESH_DATA_DIR
    / TIMEPOINT
    / mesh_cfg["zarr_name_by_tp"][TIMEPOINT]
    / WELL[0]
    / WELL[1:]
    / mesh_cfg["round_by_tp"][TIMEPOINT]
    / "meshes"
    / mesh_cfg["meshname_by_tp"][TIMEPOINT]
    / f"{ORGANOID_ID}.vtp"
)

if not mesh_path.exists():
    raise FileNotFoundError(mesh_path)

print("mesh      =", mesh_path)


mesh      = /home/fmoller/Projects/LearningOrganoids/NicoleData/20251201/fractal_output/day4p5/251130R0.zarr/B/03/2_zillum_registered/meshes/nnorg_corrected_smoothed_annotated_by_projection/144.vtp


In [122]:
# Load, normalize, spectrally smooth, then compute HKS on the smoothed mesh.
mesh = OrganoidMesh(str(mesh_path))
mesh.normalize_inplace()
mesh.label_uid = f"{TIMEPOINT}_{WELL}_{ORGANOID_ID}"

# First eigendecomposition is used for the coordinate low-pass reconstruction.
mesh._eig_decomp(k=max(EIGENMODES, int(SMOOTH_LMAX**2)))
mesh.compute_spectral_coefficients(lmax=SMOOTH_LMAX)
mesh.v = np.asarray(mesh.reconstruct_from_coeffs(mesh.coeffs_v, lmax=SMOOTH_LMAX), dtype=float)

# Geometry changed, so discard spectral operators and rebuild them on the smoothed mesh.
mesh.laplacian = None
mesh.mass_matrix = None
mesh.eigvals = None
mesh.eigvecs = None
mesh.coeffs_v = None
mesh.lmax = None
mesh._eig_decomp(k=EIGENMODES)

vertex_areas = np.asarray(mesh.vertex_areas(), float)
surface_area = float(np.sum(vertex_areas))
L_mesh = float(np.sqrt(surface_area))
ts_mesh = np.geomspace(float(HKS_T_MIN), float(HKS_T_MAX), int(HKS_N_TIMES))

hks_raw = compute_hks(mesh, ts_mesh, coeffs=False)
hks_mean = np.mean(hks_raw, axis=0, keepdims=True)
hks_centered = hks_raw - hks_mean
hks_meanratio = hks_raw / hks_mean - 1.0

hks_t1 = compute_hks(mesh, [1.0], coeffs=False)[:, 0]
hks_t1_centered = hks_t1 - float(np.mean(hks_t1))

print("vertices     =", mesh.v.shape[0])
print("faces        =", mesh.f.shape[0])
print("surface area =", surface_area)
print("L_mesh      =", L_mesh)
print("smooth lmax =", SMOOTH_LMAX)
print("n_times     =", ts_mesh.size)


vertices     = 18001
faces        = 35998
surface area = 916.9456219453772
L_mesh      = 30.281109985358484
smooth lmax = 12
n_times     = 60


In [123]:
def farthest_point_sample_vertices(vertices, n_points, seed=0):
    """Simple Euclidean farthest-point sampling over mesh vertices."""
    vertices = np.asarray(vertices, float)
    n_vertices = vertices.shape[0]
    if n_points >= n_vertices:
        return np.arange(n_vertices, dtype=int)

    rng = np.random.default_rng(seed)
    selected = [int(rng.integers(n_vertices))]
    min_dist2 = np.sum((vertices - vertices[selected[0]]) ** 2, axis=1)

    for _ in range(1, int(n_points)):
        next_idx = int(np.argmax(min_dist2))
        selected.append(next_idx)
        dist2 = np.sum((vertices - vertices[next_idx]) ** 2, axis=1)
        min_dist2 = np.minimum(min_dist2, dist2)

    return np.asarray(selected, dtype=int)


sample_vertex_idx = farthest_point_sample_vertices(
    mesh.v,
    N_SAMPLE_POINTS,
    seed=SAMPLE_RANDOM_SEED,
)

sample_colors = plotly_colors.sample_colorscale(
    "Turbo",
    np.linspace(0.0, 1.0, sample_vertex_idx.size),
)

for i, vid in enumerate(sample_vertex_idx):
    print(f"{i:02d}: vertex {int(vid):5d}  xyz={mesh.v[vid]}")


00: vertex 17009  xyz=[-3.45017179  0.59727918  3.69063369]
01: vertex  5160  xyz=[  9.22912422 -13.75575966  -1.18396402]
02: vertex  4604  xyz=[-2.81613709 14.78889503 -1.75760812]
03: vertex   364  xyz=[ 3.33628073 -4.45294833 -3.80050829]
04: vertex 10637  xyz=[4.85326792 8.38950897 0.5773125 ]
05: vertex  6643  xyz=[-8.91751786  7.22270401 -0.91551749]
06: vertex 11747  xyz=[-5.18632299 -8.50797222  1.26119942]
07: vertex   960  xyz=[-0.74468326  3.56366971 -3.78744569]
08: vertex 15822  xyz=[-2.38010389  8.07183408  2.90495954]
09: vertex  7528  xyz=[  2.26895701 -11.14240748  -0.54799339]
10: vertex 11920  xyz=[3.02457483 0.9685464  1.35292075]
11: vertex  3739  xyz=[-3.11040773 -2.65196031 -2.38668841]
12: vertex 11641  xyz=[ 7.72863144 -7.82990533  1.07490812]
13: vertex  1488  xyz=[-3.72318471  9.04663639 -3.44326809]
14: vertex  5477  xyz=[-5.89937875  2.38520402 -1.38709174]
15: vertex 12787  xyz=[-6.42225742 11.76528444  1.50706016]
16: vertex 13083  xyz=[ 1.10103377 12.46

## HKS normalization overview

The mesh panel is colored by `HKS(t=1) - mean(HKS(t=1))` on the smoothed mesh. The colored points mark the sampled vertices, and the same colors are used for their mean-subtracted and mean-ratio HKS curves. Legends are hidden so this remains readable when sampling 100 or more points.


In [124]:
mesh_fig = plot_organoid_mesh(
    mesh,
    vertex_values=hks_t1_centered,
    backend="plotly",
    colorscale="RdBu_r",
    center_at_zero=True,
    alpha=0.72,
    show_colorbar=True,
)
mesh_trace = mesh_fig.data[0]
mesh_trace.name = "HKS(t=1) - mesh mean"
mesh_trace.colorbar = dict(title="HKS - mean", len=0.36, x=0.43, y=0.78)

fig = make_subplots(
    rows=1,
    cols=3,
    specs=[
        [{"type": "scene"}, {"type": "xy"}, {"type": "xy"}],
    ],
    subplot_titles=(
        "Mesh: HKS(t=1) - mesh mean",
        "HKS - mesh mean HKS",
        "HKS / mesh mean HKS - 1",
    ),
    horizontal_spacing=0.06,
)

fig.add_trace(mesh_trace, row=1, col=1)

sample_xyz = np.asarray(mesh.v)[sample_vertex_idx]
fig.add_trace(
    go.Scatter3d(
        x=sample_xyz[:, 0],
        y=sample_xyz[:, 1],
        z=sample_xyz[:, 2],
        mode="markers",
        marker=dict(size=3.5, color=sample_colors, line=dict(width=0.5, color="black")),
        text=[f"{i}: v{int(vid)}" for i, vid in enumerate(sample_vertex_idx)],
        hovertemplate="%{text}<extra></extra>",
        showlegend=False,
    ),
    row=1,
    col=1,
)

line_width = 0.65
for i, vid in enumerate(sample_vertex_idx):
    hover = f"{i}: v{int(vid)}"
    fig.add_trace(
        go.Scatter(
            x=ts_mesh,
            y=hks_centered[vid],
            mode="lines",
            line=dict(color=sample_colors[i], width=line_width),
            text=[hover] * ts_mesh.size,
            hovertemplate="%{text}<br>t=%{x:.4g}<br>value=%{y:.4g}<extra></extra>",
            showlegend=False,
        ),
        row=1,
        col=2,
    )
    fig.add_trace(
        go.Scatter(
            x=ts_mesh,
            y=hks_meanratio[vid],
            mode="lines",
            line=dict(color=sample_colors[i], width=line_width),
            text=[hover] * ts_mesh.size,
            hovertemplate="%{text}<br>t=%{x:.4g}<br>value=%{y:.4g}<extra></extra>",
            showlegend=False,
        ),
        row=1,
        col=3,
    )

for row, col in [(1, 2), (1, 3)]:
    fig.update_xaxes(type="log", title_text="raw HKS time", showgrid=True, row=row, col=col)

fig.update_yaxes(title_text="HKS - mean", row=1, col=2)
fig.update_yaxes(title_text="HKS / mean - 1", row=1, col=3)

fig.update_layout(
    title=f"{mesh.label_uid}: sampled HKS normalization curves",
    width=1650,
    height=620,
    showlegend=False,
    template="plotly_white",
)
fig.update_layout(
    scene=dict(
        xaxis=dict(visible=False),
        yaxis=dict(visible=False),
        zaxis=dict(visible=False),
        bgcolor="rgba(0,0,0,0)",
        aspectmode="data",
    )
)

display(fig)


## Seeded connected curve-similarity segmentation

This prototype first finds seed vertices at local maxima of the `HKS / mean - 1` peak height, then grows connected regions whose full HKS/mean curves remain similar to each region's current average curve. The HKS curves themselves are not smoothed. Late timepoints that have nearly equilibrated contribute little because curve-distance weights are proportional to across-vertex variance at each time.

In [125]:
# --- Tunable prototype parameters ---
SEED_PEAK_QUANTILE = 0.65
SEED_NMS_RINGS = 2
MAX_SEEDS = 10
REGION_GROW_EDGE_QUANTILE = 0.95
REGION_GROW_DISTANCE = None  # set a float to override the edge-quantile threshold
AMPLITUDE_DISTANCE_WEIGHT = 0.35
MIN_SEEDED_REGION_AREA_FRAC = 0.001
MIN_BACKGROUND_REGION_AREA_FRAC = 0.0010
COMPACT_GROWTH = True
COMPACTNESS_MIN_AREA_OVER_DISK = 0.28
COMPACTNESS_MIN_VERTICES = 8


def vertex_adjacency_from_faces(n_vertices, faces):
    adj = [set() for _ in range(int(n_vertices))]
    for a, b, c in np.asarray(faces, dtype=int):
        adj[a].update((b, c))
        adj[b].update((a, c))
        adj[c].update((a, b))
    return [np.asarray(sorted(s), dtype=int) for s in adj]


def connected_components_from_mask(adj, mask):
    mask = np.asarray(mask, dtype=bool)
    visited = np.zeros(mask.size, dtype=bool)
    components = []
    for start in np.flatnonzero(mask):
        if visited[start]:
            continue
        stack = [int(start)]
        visited[start] = True
        comp = []
        while stack:
            u = stack.pop()
            comp.append(u)
            for v in adj[u]:
                if mask[v] and not visited[v]:
                    visited[v] = True
                    stack.append(int(v))
        components.append(np.asarray(comp, dtype=int))
    return components


def graph_ball(adj, seed, rings):
    seen = {int(seed)}
    frontier = {int(seed)}
    for _ in range(int(rings)):
        nxt = set()
        for u in frontier:
            nxt.update(map(int, adj[u]))
        nxt -= seen
        seen |= nxt
        frontier = nxt
        if not frontier:
            break
    return seen


def unique_edges_from_faces(faces):
    f = np.asarray(faces, dtype=int)
    edges = np.vstack((f[:, [0, 1]], f[:, [1, 2]], f[:, [2, 0]]))
    edges.sort(axis=1)
    return np.unique(edges, axis=0)


def edge_length_lookup(vertices, adj):
    lengths = []
    for u, nbrs in enumerate(adj):
        lengths.append({
            int(v): float(np.linalg.norm(vertices[int(u)] - vertices[int(v)]))
            for v in nbrs
        })
    return lengths


n_vertices = mesh.v.shape[0]
adj = vertex_adjacency_from_faces(n_vertices, mesh.f)
edges = unique_edges_from_faces(mesh.f)
edge_lengths = edge_length_lookup(mesh.v, adj)

seg_curves = np.asarray(hks_meanratio, float)
time_variance = np.var(seg_curves, axis=0)
if np.sum(time_variance) <= 0:
    time_weights = np.full(ts_mesh.size, 1.0 / ts_mesh.size)
else:
    time_weights = time_variance / np.sum(time_variance)
sqrt_time_weights = np.sqrt(time_weights)
weighted_curves = seg_curves * sqrt_time_weights[None, :]

shape_curves = weighted_curves - np.mean(weighted_curves, axis=1, keepdims=True)
shape_norms = np.linalg.norm(shape_curves, axis=1, keepdims=True)
shape_unit = shape_curves / np.maximum(shape_norms, np.finfo(float).eps)

mean_weighted_curve = np.mean(weighted_curves, axis=0, keepdims=True)
curve_scale = np.sqrt(np.mean(np.sum((weighted_curves - mean_weighted_curve) ** 2, axis=1)))
curve_scale = max(float(curve_scale), np.finfo(float).eps)

edge_shape_dist = 1.0 - np.sum(shape_unit[edges[:, 0]] * shape_unit[edges[:, 1]], axis=1)
edge_amp_dist = np.linalg.norm(weighted_curves[edges[:, 0]] - weighted_curves[edges[:, 1]], axis=1) / curve_scale
edge_curve_dist = edge_shape_dist + AMPLITUDE_DISTANCE_WEIGHT * edge_amp_dist
grow_distance_threshold = (
    float(np.quantile(edge_curve_dist, REGION_GROW_EDGE_QUANTILE))
    if REGION_GROW_DISTANCE is None
    else float(REGION_GROW_DISTANCE)
)

peak_idx = np.argmax(seg_curves, axis=1)
peak_height = seg_curves[np.arange(n_vertices), peak_idx]
peak_time = ts_mesh[peak_idx]
seed_threshold = max(0.0, float(np.quantile(peak_height, SEED_PEAK_QUANTILE)))

local_maxima = []
for v in np.flatnonzero(peak_height >= seed_threshold):
    nbrs = adj[int(v)]
    if nbrs.size == 0 or peak_height[v] >= np.max(peak_height[nbrs]):
        local_maxima.append(int(v))

local_maxima = sorted(local_maxima, key=lambda v: peak_height[v], reverse=True)
blocked = np.zeros(n_vertices, dtype=bool)
seed_vertices = []
for v in local_maxima:
    if blocked[v]:
        continue
    seed_vertices.append(v)
    for u in graph_ball(adj, v, SEED_NMS_RINGS):
        blocked[u] = True
    if len(seed_vertices) >= int(MAX_SEEDS):
        break

if len(seed_vertices) == 0:
    seed_vertices = [int(np.argmax(peak_height))]


labels = -np.ones(n_vertices, dtype=int)
region_weighted_sum = []
region_area_sum = []
region_vertex_count = []
region_path_radius = []
best_path_radius = []
heap = []

for label, seed in enumerate(seed_vertices):
    labels[seed] = label
    area = float(vertex_areas[seed])
    region_weighted_sum.append(weighted_curves[seed] * area)
    region_area_sum.append(area)
    region_vertex_count.append(1)
    region_path_radius.append(0.0)
    best_path_radius.append({int(seed): 0.0})

for label, seed in enumerate(seed_vertices):
    for nbr in adj[seed]:
        path_radius = edge_lengths[int(seed)][int(nbr)]
        best_path_radius[label][int(nbr)] = path_radius
        heapq.heappush(heap, (0.0, label, int(nbr), path_radius))


def distance_to_region(vertex, label):
    centroid = region_weighted_sum[label] / max(region_area_sum[label], np.finfo(float).eps)
    centroid_shape = centroid - np.mean(centroid)
    centroid_shape /= max(float(np.linalg.norm(centroid_shape)), np.finfo(float).eps)
    shape_dist = 1.0 - float(np.dot(shape_unit[vertex], centroid_shape))
    amp_dist = float(np.linalg.norm(weighted_curves[vertex] - centroid) / curve_scale)
    return shape_dist + AMPLITUDE_DISTANCE_WEIGHT * amp_dist


def passes_compact_growth(label, candidate_area, candidate_radius):
    if not COMPACT_GROWTH:
        return True
    if region_vertex_count[label] + 1 < int(COMPACTNESS_MIN_VERTICES):
        return True
    disk_area = np.pi * max(float(candidate_radius), np.finfo(float).eps) ** 2
    return candidate_area / disk_area >= float(COMPACTNESS_MIN_AREA_OVER_DISK)


while heap:
    _, label, vertex, path_radius = heapq.heappop(heap)
    if labels[vertex] != -1:
        continue
    if path_radius > best_path_radius[label].get(vertex, np.inf) + np.finfo(float).eps:
        continue
    dist = distance_to_region(vertex, label)
    if dist > grow_distance_threshold:
        continue

    area = float(vertex_areas[vertex])
    candidate_area = region_area_sum[label] + area
    candidate_radius = max(region_path_radius[label], path_radius)
    if not passes_compact_growth(label, candidate_area, candidate_radius):
        continue

    labels[vertex] = label
    region_weighted_sum[label] += weighted_curves[vertex] * area
    region_area_sum[label] = candidate_area
    region_vertex_count[label] += 1
    region_path_radius[label] = candidate_radius

    for nbr in adj[vertex]:
        if labels[nbr] == -1:
            nbr = int(nbr)
            nbr_path_radius = path_radius + edge_lengths[int(vertex)][nbr]
            if nbr_path_radius < best_path_radius[label].get(nbr, np.inf):
                best_path_radius[label][nbr] = nbr_path_radius
                heapq.heappush(heap, (dist, label, nbr, nbr_path_radius))

# Drop tiny seeded regions back into the unassigned pool.
min_seeded_area = float(MIN_SEEDED_REGION_AREA_FRAC) * surface_area
for label in range(len(seed_vertices)):
    idx = np.flatnonzero(labels == label)
    if idx.size and float(np.sum(vertex_areas[idx])) < min_seeded_area:
        labels[idx] = -1

# Remaining connected components are retained as unlabeled/background regions.
min_background_area = float(MIN_BACKGROUND_REGION_AREA_FRAC) * surface_area
next_label = int(labels.max()) + 1
for comp in connected_components_from_mask(adj, labels == -1):
    comp_area = float(np.sum(vertex_areas[comp]))
    if comp_area >= min_background_area:
        labels[comp] = next_label
        next_label += 1
    else:
        neighbor_labels = labels[np.concatenate([adj[v] for v in comp if adj[v].size > 0])]
        neighbor_labels = neighbor_labels[neighbor_labels >= 0]
        if neighbor_labels.size:
            labels[comp] = np.bincount(neighbor_labels).argmax()
        else:
            labels[comp] = next_label
            next_label += 1

# Reorder regions by descending area for stable colors and summaries.
old_region_ids = np.asarray(sorted(np.unique(labels)), dtype=int)
old_region_areas = np.asarray([np.sum(vertex_areas[labels == rid]) for rid in old_region_ids], dtype=float)
ordered_old_ids = old_region_ids[np.argsort(old_region_areas)[::-1]]
remap = {int(old): int(new) for new, old in enumerate(ordered_old_ids)}
region_labels = np.asarray([remap[int(x)] for x in labels], dtype=int)
region_ids = np.arange(len(ordered_old_ids), dtype=int)
region_vertex_sets = [np.flatnonzero(region_labels == rid) for rid in region_ids]
region_areas = np.asarray([np.sum(vertex_areas[idx]) for idx in region_vertex_sets], dtype=float)
region_compactness = []
for idx, area in zip(region_vertex_sets, region_areas):
    center = np.average(mesh.v[idx], weights=vertex_areas[idx], axis=0)
    radius = np.max(np.linalg.norm(mesh.v[idx] - center, axis=1))
    region_compactness.append(area / (np.pi * max(float(radius), np.finfo(float).eps) ** 2))
region_compactness = np.asarray(region_compactness, dtype=float)
region_seed_counts = np.asarray([
    sum(region_labels[seed] == rid for seed in seed_vertices)
    for rid in region_ids
], dtype=int)

print("seed threshold          =", seed_threshold)
print("selected seeds          =", len(seed_vertices))
print("grow distance threshold =", grow_distance_threshold)
print("compact growth          =", COMPACT_GROWTH)
print("min area / pi r^2       =", COMPACTNESS_MIN_AREA_OVER_DISK)
print("regions                 =", len(region_ids))
for rid, area, nseed, compactness in zip(region_ids, region_areas, region_seed_counts, region_compactness):
    print(f"R{rid:02d}: area={area:.4f} ({area / surface_area:.2%}), seeds={int(nseed)}, compact={compactness:.2f}")


seed threshold          = 0.24912535109259903
selected seeds          = 4
grow distance threshold = 0.057518038615033726
compact growth          = True
min area / pi r^2       = 0.28
regions                 = 5
R00: area=825.0053 (89.97%), seeds=0, compact=0.88
R01: area=53.0408 (5.78%), seeds=1, compact=0.94
R02: area=19.3168 (2.11%), seeds=1, compact=1.08
R03: area=11.5964 (1.26%), seeds=1, compact=1.12
R04: area=7.9863 (0.87%), seeds=1, compact=0.83


In [126]:
region_colors = plotly_colors.sample_colorscale(
    "Turbo",
    np.linspace(0.0, 1.0, max(len(region_ids), 2))[: len(region_ids)],
)
region_names = [
    f"R{rid}: area={region_areas[rid] / surface_area:.1%}, seeds={int(region_seed_counts[rid])}"
    for rid in region_ids
]

fig_regions = plot_mesh_by_regions(
    mesh,
    region_vertex_sets,
    backend="plotly",
    region_names=region_names,
    colors=region_colors,
    alpha=1.0,
    add_legend=False,
    fig_size=(850, 650),
)
seed_xyz = np.asarray(mesh.v)[seed_vertices]
fig_regions.add_trace(
    go.Scatter3d(
        x=seed_xyz[:, 0],
        y=seed_xyz[:, 1],
        z=seed_xyz[:, 2],
        mode="markers",
        marker=dict(size=4.5, color="black", symbol="diamond"),
        text=[f"seed {i}: v{int(v)}<br>peak={peak_height[v]:.3g}<br>t={peak_time[v]:.3g}" for i, v in enumerate(seed_vertices)],
        hovertemplate="%{text}<extra></extra>",
        showlegend=False,
        name="seeds",
    )
)
fig_regions.update_layout(title=f"{mesh.label_uid}: seeded connected HKS-curve regions")
display(fig_regions)

fig_region_curves = go.Figure()
for rid, idx in enumerate(region_vertex_sets):
    weights = vertex_areas[idx]
    region_curve = np.average(hks_meanratio[idx], axis=0, weights=weights)
    fig_region_curves.add_trace(
        go.Scatter(
            x=ts_mesh,
            y=region_curve,
            mode="lines",
            line=dict(color=region_colors[rid], width=2.0),
            name=f"R{rid}",
            text=[region_names[rid]] * ts_mesh.size,
            hovertemplate="%{text}<br>t=%{x:.4g}<br>HKS/mean-1=%{y:.4g}<extra></extra>",
            showlegend=False,
        )
    )

fig_region_curves.update_xaxes(type="log", title_text="raw HKS time")
fig_region_curves.update_yaxes(title_text="area-weighted mean HKS / mesh mean - 1")
fig_region_curves.update_layout(
    title="Region-average HKS/mean curves",
    width=850,
    height=450,
    template="plotly_white",
    showlegend=False,
)
display(fig_region_curves)


## Ellipsoid body-reference prototype

This section fits a soft-barrier ellipsoid to the smoothed organoid as a coarse body/villus estimate. The fitted ellipsoid is converted into a same-topology mesh by radial projection from the fitted center, HKS is computed on that ellipsoid mesh, and candidate bumps are scored by combining positive height over the ellipsoid with HKS excess over the ellipsoid reference.

In [130]:
# --- Tunable prototype parameters ---
ELLIPSOID_BARRIER_WEIGHT = 100.0      # Penalty for vertices that the ellipsoid would place outside the organoid; higher keeps the fit more internal.
ELLIPSOID_UNDERFILL_WEIGHT = 0.4    # Penalty for organoid surface lying outside the ellipsoid; higher inflates the ellipsoid toward bumps/crypts.
ELLIPSOID_CENTER_REGULARIZATION = 0.02  # Penalty for moving the ellipsoid center away from the solid center-of-mass initialization.
ELLIPSOID_CENTER_SHIFT_LIMIT_FRAC = 0.45  # Maximum center shift along each PCA axis, as a fraction of the initial radius on that axis.
ELLIPSOID_INITIAL_RADIUS_QUANTILE = 0.78  # Surface-distance quantile used to initialize ellipsoid radii before optimization.
ELLIPSOID_MAXITER = 1200             # Maximum optimizer iterations for the ellipsoid fit and fallback retry.


def solid_center_of_mass(vertices, faces):
    tri = np.asarray(vertices, float)[np.asarray(faces, int)]
    a = tri[:, 0]
    b = tri[:, 1]
    c = tri[:, 2]
    signed_volume = np.einsum("ij,ij->i", a, np.cross(b, c)) / 6.0
    total_volume = float(np.sum(signed_volume))
    if abs(total_volume) <= np.finfo(float).eps:
        return np.average(vertices, weights=vertex_areas, axis=0), 0.0
    centroid = np.sum(signed_volume[:, None] * (a + b + c) / 4.0, axis=0) / total_volume
    return centroid, abs(total_volume)


def signed_distance_to_mesh(points, vertices, faces):
    points = np.atleast_2d(np.asarray(points, dtype=float))
    return igl.signed_distance(
        points,
        np.asarray(vertices, dtype=float),
        np.asarray(faces, dtype=np.int64),
        igl.SIGNED_DISTANCE_TYPE_WINDING_NUMBER,
    )


def inside_sign_calibration(vertices, faces):
    center = np.mean(vertices, axis=0)
    span = np.linalg.norm(np.ptp(vertices, axis=0))
    direction = np.array([1.0, 0.37, -0.23])
    direction /= np.linalg.norm(direction)
    far_point = center + 4.0 * span * direction
    s_far, _, _, _ = signed_distance_to_mesh(far_point, vertices, faces)
    outside_sign = np.sign(float(s_far[0]))
    if outside_sign == 0:
        outside_sign = 1.0
    return outside_sign


OUTSIDE_SIGN = inside_sign_calibration(mesh.v, mesh.f)


def point_is_inside_mesh(point, vertices=mesh.v, faces=mesh.f):
    s, _, _, _ = signed_distance_to_mesh(point, vertices, faces)
    return np.sign(float(s[0])) != OUTSIDE_SIGN


def move_point_inside_mesh(point, vertices, faces):
    point = np.asarray(point, dtype=float)
    if point_is_inside_mesh(point, vertices, faces):
        return point, False

    _, _, closest, _ = signed_distance_to_mesh(point, vertices, faces)
    closest = closest[0]
    scale = float(np.linalg.norm(np.ptp(vertices, axis=0)))
    hints = [np.mean(vertices, axis=0), np.average(vertices, weights=vertex_areas, axis=0)]
    for hint in hints:
        direction = np.asarray(hint, dtype=float) - closest
        norm = float(np.linalg.norm(direction))
        if norm <= np.finfo(float).eps:
            continue
        direction /= norm
        for eps in (1e-5, 1e-4, 1e-3, 1e-2):
            candidate = closest + eps * scale * direction
            if point_is_inside_mesh(candidate, vertices, faces):
                return candidate, True
    return closest, True


def weighted_pca_frame(points, weights, center):
    centered = np.asarray(points, float) - np.asarray(center, float)[None, :]
    weights = np.asarray(weights, float)
    cov = (centered * weights[:, None]).T @ centered / max(float(np.sum(weights)), np.finfo(float).eps)
    evals, evecs = np.linalg.eigh(cov)
    order = np.argsort(evals)[::-1]
    axes = evecs[:, order]
    if np.linalg.det(axes) < 0:
        axes[:, -1] *= -1.0
    return axes


def ellipsoid_level(points, center, axes, radii):
    local = (np.asarray(points, float) - np.asarray(center, float)[None, :]) @ axes
    return np.sqrt(np.sum((local / radii[None, :]) ** 2, axis=1))


def project_points_to_ellipsoid(points, center, axes, radii):
    local = (np.asarray(points, float) - np.asarray(center, float)[None, :]) @ axes
    level = np.sqrt(np.sum((local / radii[None, :]) ** 2, axis=1))
    level = np.maximum(level, np.finfo(float).eps)
    projected_local = local / level[:, None]
    return center[None, :] + projected_local @ axes.T, level


def fit_soft_barrier_ellipsoid(vertices, faces, areas):
    com, solid_volume = solid_center_of_mass(vertices, faces)
    center0, moved_inside = move_point_inside_mesh(com, vertices, faces)
    axes = weighted_pca_frame(vertices, areas, center0)
    local0 = (vertices - center0[None, :]) @ axes
    initial_radii = np.quantile(np.abs(local0), ELLIPSOID_INITIAL_RADIUS_QUANTILE, axis=0) * 1.25
    max_radii = np.max(np.abs(local0), axis=0)
    min_radius = 0.02 * max(float(np.linalg.norm(np.ptp(vertices, axis=0))), np.finfo(float).eps)
    initial_radii = np.maximum(initial_radii, min_radius)
    max_radii = np.maximum(max_radii, initial_radii)
    area_weights = areas / max(float(np.sum(areas)), np.finfo(float).eps)
    shift_limit = ELLIPSOID_CENTER_SHIFT_LIMIT_FRAC * initial_radii

    def unpack(params):
        shift_local = params[:3]
        radii = np.exp(params[3:])
        center = center0 + shift_local @ axes.T
        return center, radii, shift_local

    def objective(params):
        center, radii, shift_local = unpack(params)
        level = ellipsoid_level(vertices, center, axes, radii)
        residual = level - 1.0
        weights = np.where(residual < 0.0, ELLIPSOID_BARRIER_WEIGHT, ELLIPSOID_UNDERFILL_WEIGHT)
        data_loss = np.sum(area_weights * weights * residual**2) / max(float(np.sum(area_weights * weights)), np.finfo(float).eps)
        shift_loss = ELLIPSOID_CENTER_REGULARIZATION * np.sum((shift_local / np.maximum(initial_radii, np.finfo(float).eps)) ** 2)
        return float(data_loss + shift_loss)

    x0 = np.r_[np.zeros(3), np.log(initial_radii)]
    bounds = [(-shift_limit[i], shift_limit[i]) for i in range(3)]
    bounds += [(np.log(0.25 * initial_radii[i]), np.log(1.35 * max_radii[i])) for i in range(3)]
    result = minimize(objective, x0, method="L-BFGS-B", bounds=bounds, options={"maxiter": int(ELLIPSOID_MAXITER)})
    if not result.success:
        retry = minimize(
            objective,
            result.x,
            method="Powell",
            bounds=bounds,
            options={"maxiter": int(ELLIPSOID_MAXITER), "xtol": 1e-5, "ftol": 1e-6},
        )
        if retry.success or retry.fun <= result.fun * (1.0 + 1e-5):
            result = retry
    center, radii, shift_local = unpack(result.x)
    fitted_vertices, vertex_level = project_points_to_ellipsoid(vertices, center, axes, radii)
    signed_height = np.sign(vertex_level - 1.0) * np.linalg.norm(vertices - fitted_vertices, axis=1)
    return {
        "solid_center_of_mass": com,
        "solid_volume": solid_volume,
        "center_of_mass_moved_inside": moved_inside,
        "center0": center0,
        "center": center,
        "axes": axes,
        "radii": radii,
        "shift_local": shift_local,
        "initial_radii": initial_radii,
        "result": result,
        "vertices": fitted_vertices,
        "level": vertex_level,
        "height": signed_height,
    }


ellipsoid_fit = fit_soft_barrier_ellipsoid(mesh.v, mesh.f, vertex_areas)
ellipsoid_vertices = ellipsoid_fit["vertices"]
ellipsoid_height = ellipsoid_fit["height"]

ellipsoid_mesh = OrganoidMesh().load_from_arrays(ellipsoid_vertices, mesh.f)
ellipsoid_mesh._eig_decomp(k=min(int(EIGENMODES), ellipsoid_vertices.shape[0] - 2))
ellipsoid_vertex_areas = np.asarray(ellipsoid_mesh.vertex_areas(), float)
ellipsoid_hks_raw = compute_hks(ellipsoid_mesh, ts_mesh, coeffs=False)
ellipsoid_hks_mean = np.mean(ellipsoid_hks_raw, axis=0, keepdims=True)
ellipsoid_hks_meanratio = ellipsoid_hks_raw / ellipsoid_hks_mean - 1.0
ellipsoid_hks_excess = hks_meanratio - ellipsoid_hks_meanratio

print("solid volume                =", ellipsoid_fit["solid_volume"])
print("COM moved inside            =", ellipsoid_fit["center_of_mass_moved_inside"])
print("fit success                 =", bool(ellipsoid_fit["result"].success))
print("fit status                  =", int(ellipsoid_fit["result"].status))
print("fit message                 =", ellipsoid_fit["result"].message)
print("fit objective               =", float(ellipsoid_fit["result"].fun))
print("initial radii               =", np.round(ellipsoid_fit["initial_radii"], 4))
print("fitted radii                =", np.round(ellipsoid_fit["radii"], 4))
print("center shift in PCA frame   =", np.round(ellipsoid_fit["shift_local"], 4))
print("height range                =", np.round(np.quantile(ellipsoid_height, [0.01, 0.5, 0.99]), 4))


solid volume                = 1346.3231751336066
COM moved inside            = False
fit success                 = True
fit status                  = 0
fit message                 = Optimization terminated successfully.
fit objective               = 0.013038496361958135
initial radii               = [14.9632  6.5962  3.7917]
fitted radii                = [7.5326 4.7663 4.3603]
center shift in PCA frame   = [ 1.3966 -0.7957 -0.1021]
height range                = [-0.9406  2.9491 11.4715]


In [135]:
# --- Ellipsoid-height + HKS-excess bump candidates ---
ELLIPSOID_BUMP_HKS_T_MAX = 75.0          # Largest HKS time scale used when scoring local HKS excess over the ellipsoid reference.
ELLIPSOID_BUMP_HEIGHT_QUANTILE = 0.4    # Minimum robust height percentile a vertex must exceed to be considered protruding above the ellipsoid.
ELLIPSOID_BUMP_HKS_QUANTILE = 0.76       # Minimum robust HKS-excess percentile a vertex must exceed to look crypt-like relative to the ellipsoid.
ELLIPSOID_BUMP_SCORE_QUANTILE = 0.82     # Minimum combined height/HKS score percentile for retaining a candidate vertex.
ELLIPSOID_BUMP_HEIGHT_WEIGHT = 0.4      # Relative contribution of positive ellipsoid height to the combined bump score.
ELLIPSOID_BUMP_HKS_WEIGHT = 0.55         # Relative contribution of HKS excess to the combined bump score.
ELLIPSOID_MIN_CANDIDATE_AREA_FRAC = 0.0010  # Smallest connected candidate area to keep, as a fraction of total mesh surface area.


def robust_zscore(values):
    values = np.asarray(values, dtype=float)
    med = float(np.nanmedian(values))
    mad = float(np.nanmedian(np.abs(values - med)))
    scale = 1.4826 * mad
    if not np.isfinite(scale) or scale <= np.finfo(float).eps:
        scale = float(np.nanstd(values))
    if not np.isfinite(scale) or scale <= np.finfo(float).eps:
        scale = 1.0
    return (values - med) / scale


bump_time_mask = ts_mesh <= float(ELLIPSOID_BUMP_HKS_T_MAX)
if not np.any(bump_time_mask):
    bump_time_mask = np.ones_like(ts_mesh, dtype=bool)

ellipsoid_hks_bump_signal = np.max(ellipsoid_hks_excess[:, bump_time_mask], axis=1)
height_z = robust_zscore(ellipsoid_height)
hks_excess_z = robust_zscore(ellipsoid_hks_bump_signal)
height_score = np.maximum(0.0, height_z)
hks_score = np.maximum(0.0, hks_excess_z)
ellipsoid_bump_score = (
    float(ELLIPSOID_BUMP_HEIGHT_WEIGHT) * height_score
    + float(ELLIPSOID_BUMP_HKS_WEIGHT) * hks_score
)

height_threshold = float(np.quantile(height_z, ELLIPSOID_BUMP_HEIGHT_QUANTILE))
hks_threshold = float(np.quantile(hks_excess_z, ELLIPSOID_BUMP_HKS_QUANTILE))
score_threshold = float(np.quantile(ellipsoid_bump_score, ELLIPSOID_BUMP_SCORE_QUANTILE))

ellipsoid_candidate_mask = (
    (height_z >= height_threshold)
    & (hks_excess_z >= hks_threshold)
    & (ellipsoid_bump_score >= score_threshold)
)

min_candidate_area = float(ELLIPSOID_MIN_CANDIDATE_AREA_FRAC) * surface_area
ellipsoid_candidate_components = []
for comp in connected_components_from_mask(adj, ellipsoid_candidate_mask):
    comp_area = float(np.sum(vertex_areas[comp]))
    if comp_area >= min_candidate_area:
        ellipsoid_candidate_components.append(comp)

ellipsoid_candidate_labels = np.zeros(n_vertices, dtype=int)
for i, comp in enumerate(ellipsoid_candidate_components, start=1):
    ellipsoid_candidate_labels[comp] = i

print("HKS bump times              =", (float(ts_mesh[bump_time_mask][0]), float(ts_mesh[bump_time_mask][-1])))
print("height z threshold          =", height_threshold)
print("HKS excess z threshold      =", hks_threshold)
print("bump score threshold        =", score_threshold)
print("candidate regions           =", len(ellipsoid_candidate_components))
for i, comp in enumerate(ellipsoid_candidate_components, start=1):
    area = float(np.sum(vertex_areas[comp]))
    print(
        f"E{i:02d}: area={area:.4f} ({area / surface_area:.2%}), "
        f"height_z={np.mean(height_z[comp]):.2f}, hks_z={np.mean(hks_excess_z[comp]):.2f}, "
        f"score={np.mean(ellipsoid_bump_score[comp]):.2f}"
    )


HKS bump times              = (0.5, 66.84318266764782)
height z threshold          = -0.30722223012523214
HKS excess z threshold      = 0.8018839300085164
bump score threshold        = 1.222143205477807
candidate regions           = 2
E01: area=86.7093 (9.46%), height_z=1.82, hks_z=1.72, score=1.67
E02: area=57.9110 (6.32%), height_z=0.35, hks_z=4.05, score=2.36


In [136]:
ellipsoid_fit_fig = make_subplots(
    rows=1,
    cols=2,
    specs=[[{"type": "scene"}, {"type": "scene"}]],
    subplot_titles=("Fitted body ellipsoid", "Height over fitted ellipsoid"),
)
height_abs = float(np.nanmax(np.abs(ellipsoid_height)))
height_abs = max(height_abs, np.finfo(float).eps)

ellipsoid_fit_fig.add_trace(
    go.Mesh3d(
        x=mesh.v[:, 0],
        y=mesh.v[:, 1],
        z=mesh.v[:, 2],
        i=mesh.f[:, 0],
        j=mesh.f[:, 1],
        k=mesh.f[:, 2],
        color="lightgray",
        opacity=0.24,
        flatshading=True,
        name="organoid mesh",
        showscale=False,
        showlegend=False,
    ),
    row=1,
    col=1,
)
ellipsoid_fit_fig.add_trace(
    go.Mesh3d(
        x=ellipsoid_vertices[:, 0],
        y=ellipsoid_vertices[:, 1],
        z=ellipsoid_vertices[:, 2],
        i=mesh.f[:, 0],
        j=mesh.f[:, 1],
        k=mesh.f[:, 2],
        color="rgb(31,119,180)",
        opacity=0.52,
        flatshading=True,
        name="fitted ellipsoid",
        showscale=False,
        showlegend=False,
    ),
    row=1,
    col=1,
)
ellipsoid_fit_fig.add_trace(
    go.Scatter3d(
        x=[ellipsoid_fit["center"][0]],
        y=[ellipsoid_fit["center"][1]],
        z=[ellipsoid_fit["center"][2]],
        mode="markers",
        marker=dict(size=5, color="black"),
        name="ellipsoid center",
        showlegend=False,
    ),
    row=1,
    col=1,
)
ellipsoid_fit_fig.add_trace(
    go.Mesh3d(
        x=mesh.v[:, 0],
        y=mesh.v[:, 1],
        z=mesh.v[:, 2],
        i=mesh.f[:, 0],
        j=mesh.f[:, 1],
        k=mesh.f[:, 2],
        intensity=ellipsoid_height,
        colorscale="RdBu_r",
        cmin=-height_abs,
        cmax=height_abs,
        colorbar=dict(title="height", x=1.0, len=0.72),
        opacity=1.0,
        flatshading=True,
        name="mesh height over ellipsoid",
        showscale=True,
        showlegend=False,
    ),
    row=1,
    col=2,
)
ellipsoid_fit_fig.add_trace(
    go.Scatter3d(
        x=[ellipsoid_fit["center"][0]],
        y=[ellipsoid_fit["center"][1]],
        z=[ellipsoid_fit["center"][2]],
        mode="markers",
        marker=dict(size=4, color="black"),
        name="ellipsoid center",
        showlegend=False,
    ),
    row=1,
    col=2,
)
ellipsoid_fit_fig.update_layout(
    height=560,
    width=1120,
    margin=dict(l=0, r=20, b=0, t=40),
    scene=dict(aspectmode="data"),
    scene2=dict(aspectmode="data"),
)
display(ellipsoid_fit_fig)

candidate_fig = plot_mesh_by_regions(
    mesh,
    ellipsoid_candidate_components,
    backend="plotly",
    colorscale="Turbo",
    baseline_color="lightgray",
    baseline_name="not candidate",
    add_legend=False,
    alpha=1.0,
)
candidate_fig.update_layout(title="Ellipsoid-height + HKS-excess candidate bumps", margin=dict(l=0, r=0, b=0, t=35))
display(candidate_fig)

score_fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=("Height vs HKS excess", "Candidate mean HKS / mesh mean - 1"),
)
score_fig.add_trace(
    go.Scatter(
        x=height_z,
        y=hks_excess_z,
        mode="markers",
        marker=dict(
            size=4,
            color=ellipsoid_bump_score,
            colorscale="Turbo",
            showscale=True,
            colorbar=dict(title="score"),
            opacity=0.75,
        ),
        showlegend=False,
    ),
    row=1,
    col=1,
)
score_fig.add_vline(x=height_threshold, line_width=1, line_dash="dash", line_color="black", row=1, col=1)
score_fig.add_hline(y=hks_threshold, line_width=1, line_dash="dash", line_color="black", row=1, col=1)

candidate_colors = plotly_colors.sample_colorscale(
    "Turbo",
    [0.5] if len(ellipsoid_candidate_components) <= 1 else np.linspace(0.0, 1.0, len(ellipsoid_candidate_components)),
)
for comp, color in zip(ellipsoid_candidate_components, candidate_colors):
    w = vertex_areas[comp]
    curve = np.average(hks_meanratio[comp], weights=w, axis=0)
    ref_curve = np.average(ellipsoid_hks_meanratio[comp], weights=w, axis=0)
    score_fig.add_trace(
        go.Scatter(x=ts_mesh, y=curve, mode="lines", line=dict(color=color, width=1.5), showlegend=False),
        row=1,
        col=2,
    )
    score_fig.add_trace(
        go.Scatter(x=ts_mesh, y=ref_curve, mode="lines", line=dict(color=color, width=1.0, dash="dot"), showlegend=False),
        row=1,
        col=2,
    )
score_fig.update_xaxes(title_text="height robust z", row=1, col=1)
score_fig.update_yaxes(title_text="HKS excess robust z", row=1, col=1)
score_fig.update_xaxes(title_text="raw HKS time", type="log", row=1, col=2)
score_fig.update_yaxes(title_text="HKS / mesh mean - 1", row=1, col=2)
score_fig.update_layout(height=430, width=1100, margin=dict(l=45, r=20, t=45, b=45))
display(score_fig)
